<a href="https://colab.research.google.com/github/AyoubElKharraf/NeRF_LRM__From_2D_to_3D/blob/main/02_check_status_nerf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import sys
import torch
import subprocess

print("=== 1. INFORMATIONS SYSTÈME & PYTHON ===")
print(f"Version Python : {sys.version.split()[0]}")
print(f"Version PyTorch : {torch.__version__}")

print("\n=== 2. VÉRIFICATION CUDA & GPU ===")
cuda_available = torch.cuda.is_available()
print(f"CUDA disponible : {cuda_available}")

if cuda_available:
    device_count = torch.cuda.device_count()
    current_device = torch.cuda.current_device()
    gpu_name = torch.cuda.get_device_name(current_device)
    vram_total = torch.cuda.get_device_properties(current_device).total_memory / (1024**3)

    print(f"Nombre de GPU : {device_count}")
    print(f"Modèle de GPU : {gpu_name}")
    print(f"VRAM Totale : {vram_total:.2f} Go")
    print(f"Version CUDA compilée avec PyTorch : {torch.version.cuda}")
    print(f"Version cuDNN : {torch.backends.cudnn.version()}")

    # Test de calcul réel sur le GPU (Allocation de tenseur)
    try:
        x = torch.rand(1000, 1000, device="cuda")
        y = torch.matmul(x, x)
        print("Test de calcul matriciel GPU : SUCCÈS ✅")
    except Exception as e:
        print(f"Test de calcul matriciel GPU : ÉCHEC ❌ ({e})")
else:
    print("ATTENTION : Aucun GPU détecté. Vérifie 'Modifier le type d'exécution' -> GPU.")

print("\n=== 3. SORTIE MATÉRIELLE NVIDIA-SMI ===")
try:
    smi_output = subprocess.check_output(["nvidia-smi"]).decode("utf-8")
    print(smi_output)
except Exception as e:
    print(f"Impossible d'exécuter nvidia-smi : {e}")

=== 1. INFORMATIONS SYSTÈME & PYTHON ===
Version Python : 3.13.18
Version PyTorch : 2.14.1+cpu

=== 2. VÉRIFICATION CUDA & GPU ===
CUDA disponible : False
ATTENTION : Aucun GPU détecté. Vérifie 'Modifier le type d'exécution' -> GPU.

=== 3. SORTIE MATÉRIELLE NVIDIA-SMI ===
Impossible d'exécuter nvidia-smi : [Errno 2] No such file or directory: 'nvidia-smi'


In [4]:
!pip install nerfstudio

In [5]:
!ns-train --help

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
I0000 00:00:1794910406.547813    2642 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794910407.602839    2642 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(

In [6]:
!pip install -q huggingface_hub

In [7]:
from huggingface_hub import hf_hub_download
import zipfile
from pathlib import Path

archive = hf_hub_download(
    repo_id="nerfbaselines/nerfbaselines-data",
    repo_type="dataset",
    filename="blender/lego.zip",
    local_dir="/content/data"
)

output_dir = Path("/content/data/blender")
output_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(archive) as zip_file:
    zip_file.extractall(output_dir)

print("Dataset téléchargé dans :", output_dir)

blender/lego.zip: reconstructing file:   0%|          |  0.00B /  180MB            

blender/lego.zip: downloading bytes:           |  0.00B            

Dataset téléchargé dans : /content/data/blender


In [8]:
!find /content/data/blender -name "transforms*.json"

/content/data/blender/lego/transforms_val.json
/content/data/blender/lego/transforms_train.json
/content/data/blender/lego/transforms_test.json


In [9]:
!ns-train nerfacto \
    --max-num-iterations 1000 \
    blender-data \
    --data /content/data/blender/lego

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
I0000 00:00:1794910460.593455    2956 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794910460.960513    2956 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(

In [10]:
!find /content/data -type d -name "lego"

/content/data/blender/lego


In [11]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [12]:
!find /content/outputs -type f | head -50

/content/outputs/unnamed/nerfacto/2026-11-17_101432/config.yml


In [13]:
!find /content/outputs -name "*.ckpt"

In [14]:
!ns-render dataset --help

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
I0000 00:00:1794910496.652882    3133 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794910496.722807    3133 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(

In [15]:
import json
import shutil
from pathlib import Path

source = Path("/content/data/blender/lego")
reduced = Path("/content/data/blender/lego_one_test")

if reduced.exists():
    shutil.rmtree(reduced)

reduced.mkdir(parents=True)

for split in ["train", "val", "test"]:
    (reduced / split).symlink_to(source / split, target_is_directory=True)

for filename in ["transforms_train.json", "transforms_val.json"]:
    (reduced / filename).symlink_to(source / filename)

with open(source / "transforms_test.json") as file:
    test_transforms = json.load(file)

test_transforms["frames"] = test_transforms["frames"][:1]

with open(reduced / "transforms_test.json", "w") as file:
    json.dump(test_transforms, file)

print("Dataset réduit créé :", reduced)
print("Nombre d'images de test :", len(test_transforms["frames"]))

Dataset réduit créé : /content/data/blender/lego_one_test
Nombre d'images de test : 1


In [16]:
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-render dataset \
    --load-config /content/outputs/unnamed/nerfacto/2026-11-16_155153/config.yml \
    --data /content/data/blender/lego_one_test \
    --output-path /content/outputs/nerf_renders/lego_one_test.mp4 \
    --split test \
    --rendered-output-names rgb

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
I0000 00:00:1794910529.688230    3264 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794910529.769241    3264 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(

In [17]:
from pathlib import Path
from IPython.display import display, Image

render_dir = Path("/content/outputs/nerf_renders/lego_one_test.mp4/test")

images = sorted(
    list(render_dir.glob("*.png")) +
    list(render_dir.glob("*.jpg")) +
    list(render_dir.glob("*.jpeg"))
)

print("Images trouvées :", len(images))

for image_path in images:
    print(image_path)
    display(Image(filename=str(image_path)))

Images trouvées : 0


In [19]:
from pathlib import Path
import json
import numpy as np
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import matplotlib.pyplot as plt

dataset_dir = Path("/content/data/blender/lego")
render_path = Path(
    "/content/outputs/nerf_renders/lego_one_test.mp4/test/rgb.jpg"
)

with open(dataset_dir / "transforms_test.json") as file:
    test_data = json.load(file)

relative_path = Path(test_data["frames"][0]["file_path"])
candidate_dir = dataset_dir / relative_path.parent
candidates = sorted(candidate_dir.glob(relative_path.name + ".*"))

print("Fichiers candidats :", candidates)

if not candidates:
    raise FileNotFoundError(
        f"Aucune image trouvée pour {relative_path} dans {candidate_dir}"
    )

ground_truth_path = candidates[0]

ground_truth = np.array(Image.open(ground_truth_path).convert("RGB"))
prediction = np.array(Image.open(render_path).convert("RGB"))

if ground_truth.shape != prediction.shape:
    prediction = np.array(
        Image.fromarray(prediction).resize(
            (ground_truth.shape[1], ground_truth.shape[0])
        )
    )

psnr = peak_signal_noise_ratio(
    ground_truth,
    prediction,
    data_range=255
)

ssim = structural_similarity(
    ground_truth,
    prediction,
    channel_axis=2,
    data_range=255
)

print("Image de référence :", ground_truth_path)
print(f"PSNR : {psnr:.2f} dB")
print(f"SSIM : {ssim:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(ground_truth)
axes[0].set_title("Image de référence")
axes[0].axis("off")

axes[1].imshow(prediction)
axes[1].set_title("Rendu NeRF")
axes[1].axis("off")

plt.show()

Fichiers candidats : [PosixPath('/content/data/blender/lego/test/r_0.png')]


FileNotFoundError: [Errno 2] No such file or directory: '/content/outputs/nerf_renders/lego_one_test.mp4/test/rgb.jpg'

In [20]:
!ns-render spiral --help

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
I0000 00:00:1794910565.265292    3408 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794910565.373438    3408 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(

In [21]:
from pathlib import Path

path = Path("/usr/local/lib/python3.13/dist-packages/nerfstudio/cameras/camera_paths.py")
text = path.read_text()

old = "c2wh = torch.matmul(c2wh_global, local_c2wh)"
new = "c2wh = torch.matmul(c2wh_global, local_c2wh.to(dtype=c2wh_global.dtype))"

if old in text:
    path.write_text(text.replace(old, new, 1))
    print("Correction de la multiplication appliquée.")
elif new in text:
    print("Correction déjà présente.")
else:
    raise RuntimeError("Ligne attendue introuvable.")

Correction de la multiplication appliquée.


In [22]:
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-render spiral \
    --load-config /content/outputs/unnamed/nerfacto/2026-11-16_155153/config.yml \
    --output-path /content/outputs/nerf_renders/lego_spiral_preview.mp4 \
    --seconds 2 \
    --downscale-factor 4 \
    --rendered-output-names rgb \
    --radius 0.1

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
I0000 00:00:1794910582.811603    3488 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794910582.881301    3488 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(

In [23]:
from pathlib import Path

video = Path("/content/outputs/nerf_renders/lego_spiral_preview.mp4")

print("Existe :", video.exists())
if video.exists():
    print("Taille :", round(video.stat().st_size / 1024 / 1024, 2), "Mo")
else:
    print("La vidéo n'a pas encore été générée.")

Existe : False
La vidéo n'a pas encore été générée.


In [24]:
!grep -n "torch.cross\|torch.matmul" \
/usr/local/lib/python3.13/dist-packages/nerfstudio/cameras/camera_utils.py \
/usr/local/lib/python3.13/dist-packages/nerfstudio/cameras/camera_paths.py

/usr/local/lib/python3.13/dist-packages/nerfstudio/cameras/camera_utils.py:314:    vec0 = normalize(torch.cross(vec1_avg, vec2))
/usr/local/lib/python3.13/dist-packages/nerfstudio/cameras/camera_utils.py:315:    vec1 = normalize(torch.cross(vec2, vec0))
/usr/local/lib/python3.13/dist-packages/nerfstudio/cameras/camera_paths.py:106:        c2wh = torch.matmul(c2wh_global, local_c2wh.to(dtype=c2wh_global.dtype))


In [25]:
from pathlib import Path

# Correction 1 et 2 : camera_utils.py
path = Path("/usr/local/lib/python3.13/dist-packages/nerfstudio/cameras/camera_utils.py")
text = path.read_text()
text = text.replace(
    "vec0 = normalize(torch.cross(vec1_avg, vec2))",
    "vec0 = normalize(torch.cross(vec1_avg, vec2.to(dtype=vec1_avg.dtype), dim=-1))"
)
text = text.replace(
    "vec1 = normalize(torch.cross(vec2, vec0))",
    "vec1 = normalize(torch.cross(vec2.to(dtype=vec0.dtype), vec0, dim=-1))"
)
path.write_text(text)

# Correction 3 : camera_paths.py
path = Path("/usr/local/lib/python3.13/dist-packages/nerfstudio/cameras/camera_paths.py")
text = path.read_text()
text = text.replace(
    "c2wh = torch.matmul(c2wh_global, local_c2wh)",
    "c2wh = torch.matmul(c2wh_global, local_c2wh.to(dtype=c2wh_global.dtype))"
)
path.write_text(text)

print("Corrections appliquées.")

Corrections appliquées.


In [26]:
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-render spiral \
    --load-config /content/outputs/unnamed/nerfacto/2026-11-16_155153/config.yml \
    --output-path /content/outputs/nerf_renders/lego_spiral_preview.mp4 \
    --seconds 2 \
    --downscale-factor 4 \
    --rendered-output-names rgb \
    --radius 0.1

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
I0000 00:00:1794910600.064831    3560 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794910600.142251    3560 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(

In [27]:
from pathlib import Path

video = Path("/content/outputs/nerf_renders/lego_spiral_preview.mp4")
print("Vidéo créée :", video.exists())
if video.exists():
    print("Taille :", video.stat().st_size / 1024 / 1024, "Mo")

Vidéo créée : False


In [28]:
from pathlib import Path

video = Path("/content/outputs/nerf_renders/lego_spiral_preview.mp4")

print("Vidéo créée :", video.exists())
if video.exists():
    print("Taille :", round(video.stat().st_size / 1024 / 1024, 2), "Mo")

Vidéo créée : False


In [29]:
from IPython.display import Video, display
display(Video(str(video), embed=True))

In [30]:
from google.colab import files
files.download("/content/outputs/nerf_renders/lego_spiral_preview.mp4")

FileNotFoundError: Cannot find file: /content/outputs/nerf_renders/lego_spiral_preview.mp4

In [1]:
from pathlib import Path

config = Path("/content/outputs/unnamed/nerfacto/2026-11-16_155153/config.yml")
checkpoint = Path("/content/outputs/unnamed/nerfacto/2026-11-16_155153/nerfstudio_models/step-000000999.ckpt")
video = Path("/content/outputs/nerf_renders/lego_spiral_preview.mp4")

print("Configuration :", config.exists())
print("Checkpoint :", checkpoint.exists())
print("Vidéo :", video.exists())

Configuration : False
Checkpoint : False
Vidéo : False


In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU absent")

False
GPU absent
